<a href="https://colab.research.google.com/github/d005810/ECAA08-Manufatura-Flexivel/blob/main/etapa-01-logica/05%20-%20Formas%20Normais%20e%20Otimizacao%20Booleana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 05 - Formas Normais (FND/FNC) e Otimizador Booleano
## Linha de Triagem e Loteamento Inteligente (Manufatura Flexível)

Neste notebook desenvolvemos o motor computacional de **Formas Normais Canônicas (FND/FNC)** e implementamos o algoritmo de minimização booleana de **Quine-McCluskey** para a otimização de expressões lógicas do sistema SCADA/CLP da planta de manufatura flexível.

### 1. Implementação do Motor de Otimização e Formas Canônicas

A classe `OtimizadorBooleano` realiza:
- **Extração de Mintermos e Maxtermos:** Avaliação da tabela-verdade completa ($2^n$).
- **Forma Normal Disjuntiva Canônica (FND / SOP):** Disjunção de todos os mintermos ativos ($f=1$).
- **Forma Normal Conjuntiva Canônica (FNC / POS):** Conjunção de todos os maxtermos ($f=0$).
- **Minimização por Quine-McCluskey:** Agrupamento por peso de Hamming, combinação de termos adjacentes e seleção dos Implicantes Primos Essenciais (EPIs).
- **Minimização Dual em FNC:** Aplicação das Leis de De Morgan para obtenção da FNC mínima.

In [ ]:
import itertools
from typing import List, Dict, Callable, Tuple, Set, Optional

def formatar_tabela(dados: List[Dict]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

class OtimizadorBooleano:
    @staticmethod
    def extrair_mintermos_maxtermos(variaveis: List[str], fn_alvo: Callable[[Dict[str, bool]], bool]) -> Tuple[List[Dict[str, bool]], List[Dict[str, bool]], List[int], List[int]]:
        """Extrai mintermos (f=1) e maxtermos (f=0) avaliando a tabela-verdade completa."""
        mintermos_dict = []
        maxtermos_dict = []
        mintermos_idx = []
        maxtermos_idx = []
        
        n = len(variaveis)
        for idx, combo in enumerate(itertools.product([False, True], repeat=n)):
            env = dict(zip(variaveis, combo))
            if fn_alvo(env):
                mintermos_dict.append(env)
                mintermos_idx.append(idx)
            else:
                maxtermos_dict.append(env)
                maxtermos_idx.append(idx)
                
        return mintermos_dict, maxtermos_dict, mintermos_idx, maxtermos_idx

    @staticmethod
    def formatar_fnd_canonico(variaveis: List[str], mintermos: List[Dict[str, bool]]) -> str:
        """Gera a FND Canônica (SOP)."""
        if not mintermos:
            return "FALSO (0)"
        termos = []
        for m in mintermos:
            partes = [v if m[v] else f"not_{v}" for v in variaveis]
            termos.append("(" + " AND ".join(partes) + ")")
        return " OR \n  ".join(termos)

    @staticmethod
    def formatar_fnc_canonico(variaveis: List[str], maxtermos: List[Dict[str, bool]]) -> str:
        """Gera a FNC Canônica (POS)."""
        if not maxtermos:
            return "VERDADEIRO (1)"
        termos = []
        for M in maxtermos:
            partes = [f"not_{v}" if M[v] else v for v in variaveis]
            termos.append("(" + " OR ".join(partes) + ")")
        return " AND \n  ".join(termos)

    @staticmethod
    def _combina_termos(termo1: str, termo2: str) -> Optional[str]:
        """Combina dois termos binários se diferirem por exatamente 1 bit."""
        diff_count = 0
        res = []
        for c1, c2 in zip(termo1, termo2):
            if c1 != c2:
                diff_count += 1
                res.append('-')
            else:
                res.append(c1)
        return "".join(res) if diff_count == 1 else None

    @classmethod
    def minimizar_quine_mccluskey(cls, variaveis: List[str], mintermos_idx: List[int]) -> Tuple[List[str], str]:
        """
        Executa o algoritmo de Quine-McCluskey para encontrar a FND mínima (SOP).
        Retorna a lista de padrões dos implicantes primos essenciais e a expressão formatada.
        """
        n = len(variaveis)
        if not mintermos_idx:
            return [], "FALSO (0)"
        if len(mintermos_idx) == 2**n:
            return ['-' * n], "VERDADEIRO (1)"

        grupos = {}
        for m in mintermos_idx:
            b_str = format(m, f'0{n}b')
            num_uns = b_str.count('1')
            if num_uns not in grupos:
                grupos[num_uns] = set()
            grupos[num_uns].add((b_str, frozenset([m])))
        
        implicantes_primos = set()
        atuais = grupos

        while atuais:
            proximos = {}
            combinados = set()
            chaves = sorted(atuais.keys())

            for i in range(len(chaves) - 1):
                k1 = chaves[i]
                k2 = chaves[i + 1]
                if k2 != k1 + 1:
                    continue

                for t1, m1 in atuais[k1]:
                    for t2, m2 in atuais[k2]:
                        comb = cls._combina_termos(t1, t2)
                        if comb is not None:
                            combinados.add(t1)
                            combinados.add(t2)
                            num_uns = comb.count('1')
                            if num_uns not in proximos:
                                proximos[num_uns] = set()
                            proximos[num_uns].add((comb, m1 | m2))

            for k in atuais:
                for t, m in atuais[k]:
                    if t not in combinados:
                        implicantes_primos.add((t, m))

            atuais = proximos

        implicantes_lista = list(implicantes_primos)
        mintermos_restantes = set(mintermos_idx)
        cobertura_escolhida = []

        # Implicantes Primos Essenciais (EPI)
        for m in mintermos_idx:
            cobridores = [imp for imp in implicantes_lista if m in imp[1]]
            if len(cobridores) == 1:
                epi = cobridores[0]
                if epi not in cobertura_escolhida:
                    cobertura_escolhida.append(epi)
                    mintermos_restantes -= epi[1]

        # Heurística gulosa para cobertura dos mintermos restantes
        while mintermos_restantes:
            candidatos = [imp for imp in implicantes_lista if imp not in cobertura_escolhida]
            if not candidatos:
                break
            candidatos.sort(key=lambda imp: (len(imp[1] & mintermos_restantes), imp[0].count('-')), reverse=True)
            melhor = candidatos[0]
            cobertura_escolhida.append(melhor)
            mintermos_restantes -= melhor[1]

        padroes = [imp[0] for imp in cobertura_escolhida]
        termos_formatados = []
        for p in padroes:
            partes = []
            for var, char in zip(variaveis, p):
                if char == '1':
                    partes.append(var)
                elif char == '0':
                    partes.append(f"not_{var}")
            if not partes:
                termos_formatados.append("VERDADEIRO")
            else:
                termos_formatados.append("(" + " AND ".join(partes) + ")")

        expr_fnd_min = " OR ".join(termos_formatados) if termos_formatados else "FALSO (0)"
        return padroes, expr_fnd_min

    @classmethod
    def minimizar_fnc(cls, variaveis: List[str], maxtermos_idx: List[int]) -> Tuple[List[str], str]:
        """Minimiza a FNC aplicando Quine-McCluskey sobre maxtermos e aplicando De Morgan."""
        padroes_neg, _ = cls.minimizar_quine_mccluskey(variaveis, maxtermos_idx)
        if not padroes_neg:
            return [], "VERDADEIRO (1)"

        termos_fnc = []
        for p in padroes_neg:
            partes = []
            for var, char in zip(variaveis, p):
                if char == '1':
                    partes.append(f"not_{var}")
                elif char == '0':
                    partes.append(var)
            if partes:
                termos_fnc.append("(" + " OR ".join(partes) + ")")

        expr_fnc_min = " AND \n  ".join(termos_fnc) if termos_fnc else "VERDADEIRO (1)"
        return padroes_neg, expr_fnc_min

    @classmethod
    def calcular_metricas(cls, variaveis: List[str], mintermos_idx: List[int], padroes_otimizados: List[str]) -> Dict[str, any]:
        """Calcula métricas de literais e ganho percentual de processamento."""
        n = len(variaveis)
        num_mintermos = len(mintermos_idx)
        literais_canonicos = num_mintermos * n
        literais_otimizados = sum(n - p.count('-') for p in padroes_otimizados)
        reducao = 0.0
        if literais_canonicos > 0:
            reducao = ((literais_canonicos - literais_otimizados) / literais_canonicos) * 100
            
        return {
            "Variaveis": n,
            "Mintermos (f=1)": num_mintermos,
            "Literais Canonicos": literais_canonicos,
            "Termos Otimizados": len(padroes_otimizados),
            "Literais Otimizados": literais_otimizados,
            "Reducao Literais (%)": round(reducao, 2)
        }

print("Motor Algorítmico de Otimização Booleana carregado com sucesso!")

### 2. Estudo de Caso 1: Estação de Triagem e Classificação (Pistões XV-201, XV-202, XV-203)

No Setor 200 da Linha de Manufatura Flexível, os atuadores de triagem atuam com base nas leituras dos sensores ópticos de geometria e cor:
- `s_a` (ZS-201): Sensor óptico de base (peça presente)
- `s_b` (ZS-202): Sensor óptico de topo (peça alta/grande)
- `c_r` (AS-201): Detector de cor vermelha
- `c_g` (AS-202): Detector de cor verde
- `c_b` (AS-203): Detector de cor azul

**Condições de Atuação:**
- **Pistão 1 (`v1` / XV-201):** Peça Grande Vermelha ($s_a \land s_b \land c_r \land \neg c_g \land \neg c_b$)
- **Pistão 2 (`v2` / XV-202):** Peça Grande Verde ($s_a \land s_b \land \neg c_r \land c_g \land \neg c_b$)
- **Pistão 3 (`v3` / XV-203):** Peça Pequena Azul ($s_a \land \neg s_b \land \neg c_r \land \neg c_g \land c_b$)

In [ ]:
# Variáveis e funções de decisão da estação de triagem
vars_triagem = ['s_a', 's_b', 'c_r', 'c_g', 'c_b']

def condicao_pistao_1(env: Dict[str, bool]) -> bool:
    return env['s_a'] and env['s_b'] and env['c_r'] and (not env['c_g']) and (not env['c_b'])

def condicao_pistao_2(env: Dict[str, bool]) -> bool:
    return env['s_a'] and env['s_b'] and (not env['c_r']) and env['c_g'] and (not env['c_b'])

def condicao_pistao_3(env: Dict[str, bool]) -> bool:
    return env['s_a'] and (not env['s_b']) and (not env['c_r']) and (not env['c_g']) and env['c_b']

atuadores = [
    ("Pistão 1 (XV-201 / Peça Grande Vermelha)", condicao_pistao_1),
    ("Pistão 2 (XV-202 / Peça Grande Verde)", condicao_pistao_2),
    ("Pistão 3 (XV-203 / Peça Pequena Azul)", condicao_pistao_3),
]

metricas_triagem = []

for nome, fn in atuadores:
    min_dict, max_dict, min_idx, max_idx = OtimizadorBooleano.extrair_mintermos_maxtermos(vars_triagem, fn)
    padroes_fnd, fnd_min = OtimizadorBooleano.minimizar_quine_mccluskey(vars_triagem, min_idx)
    _, fnc_min = OtimizadorBooleano.minimizar_fnc(vars_triagem, max_idx)
    
    met = OtimizadorBooleano.calcular_metricas(vars_triagem, min_idx, padroes_fnd)
    metricas_triagem.append({"Equipamento": nome, **met})
    
    print(f"==================================================")
    print(f"--- {nome} ---")
    print(f"Mintermos Ativos (f=1): {min_idx}")
    print(f"\n[FND Canônica]:")
    print(OtimizadorBooleano.formatar_fnd_canonico(vars_triagem, min_dict))
    print(f"\n[FND Otimizada (SOP Mínima)]:")
    print(fnd_min)
    print(f"\n[FNC Otimizada (POS Mínima - Amostra dos primeiros 3 termos)]:")
    print("\n  ".join(fnc_min.split("\n  ")[:3]) + "\n  ...")
    print("\n")

print("--- RESUMO DE OTIMIZAÇÃO: SETOR 200 (TRIAGEM) ---")
print(formatar_tabela(metricas_triagem))

### 3. Estudo de Caso 2: Permissivo do Motor da Esteira M-101 ($P_{\text{M-101}}$) e Lógica de Trip

O motor da esteira principal $\text{M-101}$ avalia as seguintes condições operacionais:
- `e1`: Botão de Parada de Emergência ($1 =$ Falha)
- `a1`: Alarme Geral de Produção ($1 =$ Falha)
- `batch_full`: Caixas de saída cheias ($1 =$ Todas as caixas em 10 peças)
- `auto`: Modo Automático ($1 =$ Ativo)
- `man`: Modo Manual ($1 =$ Ativo)

**Equação Lógica de Permissão de Partida:**
$$P_{\text{M-101}} = \neg e_1 \land \neg a_1 \land \neg batch\_full \land (auto \oplus man)$$

In [ ]:
# Variáveis e lógica do motor da esteira M-101
vars_esteira = ['e1', 'a1', 'batch_full', 'auto', 'man']

def permissivo_esteira_M101(env: Dict[str, bool]) -> bool:
    modo_exclusivo = env['auto'] ^ env['man']  # XOR
    seguranca_ok = (not env['e1']) and (not env['a1']) and (not env['batch_full'])
    return seguranca_ok and modo_exclusivo

def trip_esteira_M101(env: Dict[str, bool]) -> bool:
    conflito_modo = not (env['auto'] ^ env['man'])
    falha_seguranca = env['e1'] or env['a1'] or env['batch_full']
    return falha_seguranca or conflito_modo

min_dict_p, max_dict_p, min_idx_p, max_idx_p = OtimizadorBooleano.extrair_mintermos_maxtermos(vars_esteira, permissivo_esteira_M101)
padroes_esteira, fnd_min_esteira = OtimizadorBooleano.minimizar_quine_mccluskey(vars_esteira, min_idx_p)
_, fnc_min_esteira = OtimizadorBooleano.minimizar_fnc(vars_esteira, max_idx_p)

print("==================================================")
print("--- PERMISSIVO DE PARTIDA DA ESTEIRA PRINCIPAL (M-101) ---")
print(f"Total de Mintermos Ativos (f=1): {min_idx_p}")
print(f"\n[FND Canônica Completa]:")
print(OtimizadorBooleano.formatar_fnd_canonico(vars_esteira, min_dict_p))
print(f"\n[FND Otimizada (Quine-McCluskey)]:")
print(fnd_min_esteira)
print(f"\n[FNC Otimizada (Barreiras de Segurança / Interlocks)]:")
print(fnc_min_esteira)

met_esteira = OtimizadorBooleano.calcular_metricas(vars_esteira, min_idx_p, padroes_esteira)
print(f"\n--- Métricas do Permissivo M-101 ---")
for k, v in met_esteira.items():
    print(f"{k}: {v}")

### 4. Estudo de Caso 3: Supervisão Global de Inconsistências Sensoriais ($a_1$)

O alarme de produção $a_1$ (Tag **HS-302**) realiza a validação de coerência física dos sensores:
1. **Inconsistência Geométrica:** Sensor de topo ativo sem a base ($\neg s_a \land s_b$)
2. **Conflito Óptico de Cor:** Disparo simultâneo de múltiplos canais de cor ($(c_r \land c_g) \lor (c_r \land c_b) \lor (c_g \land c_b)$)

In [ ]:
# Variáveis e lógica de supervisão sensorial (Alarme a1)
vars_inspecao = ['s_a', 's_b', 'c_r', 'c_g', 'c_b']

def falha_inconsistencia_sensorial(env: Dict[str, bool]) -> bool:
    erro_geometria = (not env['s_a']) and env['s_b']
    cores_ativas = sum([env['c_r'], env['c_g'], env['c_b']])
    erro_cor = cores_ativas > 1
    return erro_geometria or erro_cor

min_dict_a, max_dict_a, min_idx_a, max_idx_a = OtimizadorBooleano.extrair_mintermos_maxtermos(vars_inspecao, falha_inconsistencia_sensorial)
padroes_alarme, fnd_min_alarme = OtimizadorBooleano.minimizar_quine_mccluskey(vars_inspecao, min_idx_a)

print("==================================================")
print("--- SUPERVISÃO DE CONSISTÊNCIA SENSORIAL (ALARME a1) ---")
print(f"Quantidade de Combinações de Falha na Tabela: {len(min_idx_a)} de {2**len(vars_inspecao)}")
print(f"\n[FND Otimizada de Disparo de Alarme]:")
print(fnd_min_alarme)

met_alarme = OtimizadorBooleano.calcular_metricas(vars_inspecao, min_idx_a, padroes_alarme)
print(f"\n--- Métricas do Alarme a1 ---")
for k, v in met_alarme.items():
    print(f"{k}: {v}")

### 5. Prova Formal de Equivalência (Tautologia) e Resumo Consolidado

Realizamos a prova de equivalência lógica formal ($f_{\text{canônica}} \leftrightarrow f_{\text{otimizada}} \equiv 1$) para todos os $2^n$ estados possíveis de cada subsistema.

In [ ]:
import itertools
from typing import List, Dict, Callable

def avaliar_padrao_sop(padroes: List[str], variaveis: List[str], env: Dict[str, bool]) -> bool:
    """Avalia uma expressão SOP minimizada para um dado estado ambiental."""
    for p in padroes:
        termo_satisfeito = True
        for var, char in zip(variaveis, p):
            if char == '1' and not env[var]:
                termo_satisfeito = False
                break
            elif char == '0' and env[var]:
                termo_satisfeito = False
                break
        if termo_satisfeito:
            return True
    return False

# Definições explícitas dos modelos da planta para execução autônoma
vars_triagem = ['s_a', 's_b', 'c_r', 'c_g', 'c_b']
def condicao_pistao_1(env: Dict[str, bool]) -> bool:
    return env['s_a'] and env['s_b'] and env['c_r'] and (not env['c_g']) and (not env['c_b'])
def condicao_pistao_2(env: Dict[str, bool]) -> bool:
    return env['s_a'] and env['s_b'] and (not env['c_r']) and env['c_g'] and (not env['c_b'])
def condicao_pistao_3(env: Dict[str, bool]) -> bool:
    return env['s_a'] and (not env['s_b']) and (not env['c_r']) and (not env['c_g']) and env['c_b']

vars_esteira = ['e1', 'a1', 'batch_full', 'auto', 'man']
def permissivo_esteira_M101(env: Dict[str, bool]) -> bool:
    return (not env['e1']) and (not env['a1']) and (not env['batch_full']) and (env['auto'] ^ env['man'])

vars_inspecao = ['s_a', 's_b', 'c_r', 'c_g', 'c_b']
def falha_inconsistencia_sensorial(env: Dict[str, bool]) -> bool:
    erro_geometria = (not env['s_a']) and env['s_b']
    cores_ativas = sum([env['c_r'], env['c_g'], env['c_b']])
    return erro_geometria or (cores_ativas > 1)

# Lista de testes de equivalência formal
testes = [
    ("Pistão 1 (XV-201)", vars_triagem, condicao_pistao_1),
    ("Pistão 2 (XV-202)", vars_triagem, condicao_pistao_2),
    ("Pistão 3 (XV-203)", vars_triagem, condicao_pistao_3),
    ("Permissivo Esteira (M-101)", vars_esteira, permissivo_esteira_M101),
    ("Supervisor de Falhas (a1)", vars_inspecao, falha_inconsistencia_sensorial)
]

resumo_final = []

print("--- PROVA FORMAL DE TAUTOLOGIA (f_original <=> f_otimizada) ---")
for nome, vars_sys, fn in testes:
    _, _, min_idx, _ = OtimizadorBooleano.extrair_mintermos_maxtermos(vars_sys, fn)
    padroes, _ = OtimizadorBooleano.minimizar_quine_mccluskey(vars_sys, min_idx)
    
    tautologia = True
    total_estados = 2**len(vars_sys)
    for combo in itertools.product([False, True], repeat=len(vars_sys)):
        env = dict(zip(vars_sys, combo))
        val_orig = fn(env)
        val_otim = avaliar_padrao_sop(padroes, vars_sys, env)
        if val_orig != val_otim:
            tautologia = False
            break
            
    assert tautologia is True, f"Falha na validação formal para {nome}"
    print(f"✓ {nome:<30}: PROVADO (Tautologia 100%) em {total_estados} combinações.")
    
    met = OtimizadorBooleano.calcular_metricas(vars_sys, min_idx, padroes)
    resumo_final.append({
        "Subsistema / Equipamento": nome,
        "Nº Vars": met["Variaveis"],
        "Mintermos (f=1)": met["Mintermos (f=1)"],
        "Literais Canônicos": met["Literais Canonicos"],
        "Literais Otimizados": met["Literais Otimizados"],
        "Redução (%)": f"{met['Reducao Literais (%)']}%"
    })

print("\n================================================================================")
print("TABELA CONSOLIDADA: IMPACTO DA OTIMIZAÇÃO NO SCAN DO SCADA/CLP")
print("================================================================================")
print(formatar_tabela(resumo_final))